In [ ]:
!pip install git+https://github.com/unslothai/unsloth-zoo.git git+https://github.com/unslothai/unsloth.git pandas numpy torch pathlib kaggle trl==0.24.0 bitsandbytes picosvg cairosvg opencv-python-headless editdistance mergekit llm_blender outlines scikit-learn weave wandb protobuf llguidance

  Cloning https://github.com/unslothai/unsloth-zoo.git to /tmp/pip-req-build-h7t0xo7i
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth-zoo.git /tmp/pip-req-build-h7t0xo7i
  Resolved https://github.com/unslothai/unsloth-zoo.git to commit a81596bdaa837f24f92c0f34f765d4e488512b84
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-pnjw5vzb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-pnjw5vzb
  Resolved https://github.com/unslothai/unsloth.git to commit 6984e118eb5b2e885136ce6621ab9ee0eb1eac40
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [ ]:
import os
from pathlib import Path


BASE_DIR = Path("/kaggle/input/competitions/dl-spring-2026-svg-generation")
# # Download data
# if not BASE_DIR.exists() or not any(BASE_DIR.iterdir()):
#     print("Downloading and unzipping data...")
#     os.makedirs(BASE_DIR, exist_ok=True)
#     print("Downloading...")
#     !kaggle competitions download -c dl-spring-2026-svg-generation
#     print("Unzipping...")
#     !unzip -q dl-spring-2026-svg-generation.zip
#     !mv sample_submission.csv /content/dl-spring-2026-svg-generation
#     !mv train.csv /content/dl-spring-2026-svg-generation
#     !mv test.csv /content/dl-spring-2026-svg-generation
# else:
#     print("SVG Data folder already exists.")



In [3]:
import pandas as pd
import re

print("Analyzing SVG Dimensions in the Training Set.")

# Load the dataset
df = pd.read_csv("/kaggle/input/competitions/dl-spring-2026-svg-generation/train.csv")

# Regex to safely find attributes inside the opening <svg ... > tag
height_pattern = re.compile(r'<svg[^>]*\sheight="([^"]+)"')
width_pattern = re.compile(r'<svg[^>]*\swidth="([^"]+)"')
viewbox_pattern = re.compile(r'<svg[^>]*\sviewBox="([^"]+)"')

heights = []
widths = []
viewboxes = []

# Scan the dataset
for svg in df['svg'].dropna():
    # We only search the first 250 characters to save time (the opening tag)
    head = svg[:250] 
    
    h_match = height_pattern.search(head)
    w_match = width_pattern.search(head)
    vb_match = viewbox_pattern.search(head)
    
    if h_match: heights.append(h_match.group(1))
    if w_match: widths.append(w_match.group(1))
    if vb_match: viewboxes.append(vb_match.group(1))

# Print the Results
print("\n--- Top 5 Heights ---")
print(pd.Series(heights).value_counts().head(5).to_string())

print("\n--- Top 5 Widths ---")
print(pd.Series(widths).value_counts().head(5).to_string())

print("\n--- Top 5 ViewBoxes ---")
print(pd.Series(viewboxes).value_counts().head(5).to_string())

total_svgs = len(df)
print(f"\nTotal SVGs scanned: {total_svgs}")

Analyzing SVG Dimensions in the Training Set.

--- Top 5 Heights ---
200.0px    40892
128         6308
400          740
200          394
100          245

--- Top 5 Widths ---
200.0px    40892
128         6308
400          733
200          384
100          245

--- Top 5 ViewBoxes ---
0.0 0.0 200.0 200.0    40892
0 0 24 24               2104
0 0 128 128              902
0 0 400 400              734
0 0 32 32                655

Total SVGs scanned: 50000


In [ ]:
import pandas as pd
import numpy as np
import concurrent.futures
import re
import os
from picosvg.svg import SVG

# Regex Compilations (Compiled outside the loop for speed) 
FLOAT_PATTERN = re.compile(r'-?\d+\.\d+')
PATH_LETTER_SPACE_PATTERN = re.compile(r'([a-zA-Z])\s+')
COMMA_SPACE_PATTERN = re.compile(r'\s*,\s*')
HEX_COLOR_PATTERN = re.compile(r'#([0-9a-fA-F])\1([0-9a-fA-F])\2([0-9a-fA-F])\3')
MULTIPLE_SPACES_PATTERN = re.compile(r'\s+')
PATH_D_ATTR_PATTERN = re.compile(r'd="([^"]+)"')

# Cleaning Helper Functions
def round_floats(match):
    # Strictly maintains your requested 2 decimal points, but strips useless trailing zeros
    return f"{float(match.group(0)):.2f}".rstrip('0').rstrip('.')

def minify_d_attr(match):
    """Cleans spaces strictly inside the path string."""
    d_str = match.group(1)
    # Remove spaces after path command letters (e.g., "M 10" -> "M10")
    d_str = PATH_LETTER_SPACE_PATTERN.sub(r'\1', d_str)
    # Remove spaces around commas (e.g., "10, 20" -> "10,20")
    d_str = COMMA_SPACE_PATTERN.sub(',', d_str)
    # Condense multiple spaces into one
    d_str = MULTIPLE_SPACES_PATTERN.sub(' ', d_str)
    return f'd="{d_str}"'

def minify_path_and_hex(svg_string):
    # Apply the aggressive space removal ONLY to the d="..." attribute
    svg_string = PATH_D_ATTR_PATTERN.sub(minify_d_attr, svg_string)
    
    # Compress hex colors globally (e.g., #FFFFFF -> #FFF)
    svg_string = HEX_COLOR_PATTERN.sub(r'#\1\2\3', svg_string)
    
    # Condense global multiple spaces safely
    svg_string = MULTIPLE_SPACES_PATTERN.sub(' ', svg_string)
    
    return svg_string

def strip_useless_tags(svg_string):
    # Remove empty defs
    svg_string = svg_string.replace("<defs/>", "").replace("<defs></defs>", "")
    
    # Remove redundant/default attributes to save tokens
    svg_string = re.sub(r'\s*fill-opacity="1\.0"', '', svg_string)
    svg_string = re.sub(r'\s*filling="0"', '', svg_string)
    svg_string = re.sub(r'\s*fill=""', '', svg_string)
    
    # SAFELY COMPRESS: Normalize height and width instead of deleting them
    # This changes height="200.0px" or width="256px" to simply height="200"
    svg_string = re.sub(r'\s*height="[0-9.]+(px)?"', ' height="200"', svg_string)
    svg_string = re.sub(r'\s*width="[0-9.]+(px)?"', ' width="200"', svg_string)
    
    # Close gaps between XML tags
    svg_string = svg_string.replace('> <', '><')
    
    return svg_string

# Main Optimization Wrapper 
def optimize_svg(raw_svg):
    if not isinstance(raw_svg, str) or not raw_svg.strip(): 
        return ""
        
    try:
        # Step A: Standardize shapes to paths using picosvg
        svg_obj = SVG.fromstring(raw_svg)
        cleaned_svg = svg_obj.topicosvg().tostring()
        
        # Step B: Apply your 2-decimal float rounding
        cleaned_svg = FLOAT_PATTERN.sub(round_floats, cleaned_svg)
        
        # Step C: Minify paths and compress hex codes
        cleaned_svg = minify_path_and_hex(cleaned_svg)
        
        # Step D: Strip useless XML bloat
        cleaned_svg = strip_useless_tags(cleaned_svg)
        
        return cleaned_svg.strip()
    except Exception:
        return ""

# Parallel Processing Execution 
def process_chunk(chunk):
    chunk['svg'] = chunk['svg'].apply(optimize_svg)
    return chunk
    
# # This process took 20 minutes
# if __name__ == "__main__":
#     print("Starting Data Diet.")
#     # df = pd.read_csv("./dl-spring-2026-svg-generation/train.csv")
#     df = pd.read_csv("/kaggle/input/competitions/dl-spring-2026-svg-generation/train.csv")
    
#     num_cores = os.cpu_count() or 4
#     print(f"At least {num_cores} Cores are available.")
#     chunks = np.array_split(df, num_cores * 4)

#     processed_chunks = []
#     with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
#         for result in executor.map(process_chunk, chunks):
#             processed_chunks.append(result)

#     final_df = pd.concat(processed_chunks)
    
#     # Filter out any SVGs that failed parsing or returned empty
#     final_df = final_df[final_df['svg'] != ""]
    
#     final_df.to_parquet("train_optimized.parquet", index=False)
#     print(f"Data Diet Complete. {len(final_df)} highly compressed rows saved.")

In [4]:
import pandas as pd

# Load your newly compressed dataset
# df = pd.read_parquet("train_optimized.parquet")
df = pd.read_parquet("/kaggle/input/datasets/jef9921/train-optimized")

print(f"Total optimized rows ready for training: {len(df)}\n")
print("==================================================")
print("       FIRST 10 SVGS FOR SANITY CHECK")
print("==================================================\n")

# Loop through the first 10 rows
for i, row in df.head(10).iterrows():
    print(f"[{i+1}/10] PROMPT: {row['prompt']}")
    print(f"SVG STRING:\n{row['svg']}\n")
    print("-" * 80 + "\n")

Total optimized rows ready for training: 49043

       FIRST 10 SVGS FOR SANITY CHECK

[1/10] PROMPT: The image features two orange squares with a microphone icon and an arrow connecting them, set against a white background.
SVG STRING:
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200" width="200"><path fill="#FF6A00" d="M93.3,21.2 L93.3,80.4 L21.2,80.4 L21.2,179.6 L120.4,179.6 L120.4,107.1 L179.1,107.1 L179.1,21.2 L93.3,21.2 ZM113.8,172.9 L27.9,172.9 L27.9,87.1 L113.7,87.1 L113.7,172.9 L113.8,172.9 ZM172.5,100.4 L120.4,100.4 L120.4,80.4 L100,80.4 L100,27.9 L172.5,27.9 L172.5,100.4 Z"/><path fill="#FF6A00" d="M143.8,44.2 C143.8,40 140.5,37.1 136.7,37.1 C132.5,37.1 129.6,40.4 129.6,44.2 L129.6,51.3 L144.2,51.3 L144.2,44.2 L143.8,44.2 ZM129.2,62.1 C129.2,66.3 132.5,69.2 136.3,69.2 C140.5,69.2 143.4,65.9 143.4,62.1 L143.4,55 L129.2,55 L129.2,62.1 Z"/><path fill="#FF6A00" d="M149.6,62.1 C149.6,68.8 144.6,74.2 138.4,75 L135,75 C128.8,74.2 123.8,68.8 123.8,62.1 L116.

In [1]:
%%writefile train_sft.py
import os
import torch
from unsloth import FastLanguageModel

import transformers.utils.hub
transformers.utils.hub.TRANSFORMERS_CACHE = os.getenv("HF_HOME", "~/.cache/huggingface/hub")
os.environ["WANDB_DISABLED"] = "true"

from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

if __name__ == "__main__":
    train_optimized_filepath = "/kaggle/input/datasets/jef9921/train-optimized/train_optimized.parquet"
    raw_dataset = load_dataset("parquet", data_files=train_optimized_filepath, split="train")
    
    # Split into Train and Eval 
    dataset_split = raw_dataset.select(range(25000)).train_test_split(test_size=500, seed=25)
    train_dataset = dataset_split["train"]
    eval_dataset = dataset_split["test"]
    
    MODEL_ID = "unsloth/Qwen2.5-Coder-3B"
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=2048,
        dtype=torch.float16, 
        load_in_4bit=True,
    )
    
    model = FastLanguageModel.get_peft_model(
        model, 
        # Increased Capacity & MLP Layers 
        r=32, 
        lora_alpha=64, 
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
        lora_dropout=0, 
        bias="none",
        use_gradient_checkpointing=True,
        random_state=25,
    )
    
    EOS = tokenizer.eos_token
    
    def format_for_sft(example):
        open_comment = "<!-" + "-"
        close_comment = "-" + "->"
        text = f"{open_comment} Description: {example['prompt']} {close_comment}\n```xml\n{example['svg']}\n```{EOS}"
        return {"text": text}
    
    train_dataset = train_dataset.map(format_for_sft)
    eval_dataset = eval_dataset.map(format_for_sft)
    
    sft_config = SFTConfig(
        output_dir="./svg-phase1-sft",
        dataset_text_field="text",
        max_seq_length=2048,
        
        # Dataset Packing 
        packing=True, 
        
        num_train_epochs=1, 
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8, 
        
        # Advanced Training Dynamics 
        learning_rate=2e-4,
        lr_scheduler_type="cosine", 
        warmup_steps=100,           
        weight_decay=0.01,          
        neftune_noise_alpha=5,      
        
        fp16=True,  
        bf16=False, 
        optim="paged_adamw_8bit", 
        logging_steps=50,
        
        # Evaluation Strategy 
        eval_strategy="steps",
        eval_steps=200, 
        
        # Save exactly when you evaluate, and keep the best one! 
        save_strategy="steps", 
        save_steps=200,
        load_best_model_at_end=True,         # Auto-reverts to the best checkpoint!
        metric_for_best_model="eval_loss",   # Uses your eval set as the judge
        greater_is_better=False,             # Because lower loss is better
        save_total_limit=2,
        ddp_find_unused_parameters=False,
    )
    
    sft_trainer = SFTTrainer(
        model=model, 
        tokenizer=tokenizer, 
        train_dataset=train_dataset, 
        eval_dataset=eval_dataset,
        args=sft_config
    )
    
    print("Starting SFT Phase 1 with Qwen2.5-Coder-3B...")
    sft_trainer.train()
    
    model.save_pretrained("./sft-lora-adapter-3B")
    tokenizer.save_pretrained("./sft-lora-adapter-3B")

Overwriting train_sft.py


In [ ]:
# !accelerate launch --num_processes 2 train_sft.py

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading base model to CPU.")
# Load the base model strictly to the CPU to avoid the 8GB VRAM limit
base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen2.5-Coder-3B", 
    torch_dtype=torch.float16,
    device_map="cpu", 
)

print("Attaching LoRA adapter.")
# Attach your Phase 1 adapter
model = PeftModel.from_pretrained(
    base_model,
    # "./sft-lora-adapter-3B",
    "/kaggle/input/datasets/jef9921/sft-lora-3b/sft-lora-adapter-3B",
    device_map="cpu",
)

print("Merging weights (this will take a while).")
# Fuse the LoRA weights directly into the base model's linear layers
merged_model = model.merge_and_unload()

print("Saving merged model to disk.")
# Export the standalone Hugging Face model
merged_model.save_pretrained("./sft-merged-model-3b")

# tokenizer = AutoTokenizer.from_pretrained("./sft-lora-adapter-3B")
tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/datasets/jef9921/sft-lora-3b/sft-lora-adapter-3B")
tokenizer.save_pretrained("./sft-merged-model-3b")

print("Merge complete! Safe for Outlines inference/GRPO.")

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Loading base model to CPU.


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

Attaching LoRA adapter.
Merging weights (this will take a while).
Saving merged model to disk.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merge complete! Safe for Outlines inference/GRPO.


In [3]:
%%writefile train_grpo.py
import os
import torch
import transformers.utils.hub
transformers.utils.hub.TRANSFORMERS_CACHE = os.getenv("HF_HOME", "~/.cache/huggingface/hub")
os.environ["WANDB_DISABLED"] = "true"

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig
from datasets import load_dataset
import io
import cv2
import cairosvg
import numpy as np
from PIL import Image
import editdistance
import xml.etree.ElementTree as ET
from skimage.metrics import structural_similarity as ssim

from transformers.trainer_utils import get_last_checkpoint

# ==========================================
# REWARD FUNCTIONS
# ==========================================
import re

def extract_svg(text):
    """Safely extracts the SVG from markdown backticks."""
    match = re.search(r'(<svg.*?</svg>)', str(text), re.IGNORECASE | re.DOTALL)
    return match.group(1) if match else str(text)

def render_to_numpy(svg_string):
    try:
        svg_string = extract_svg(svg_string) # Clean the string first!
        if not isinstance(svg_string, str) or "<svg" not in svg_string: return None
        # png_data = cairosvg.svg2png(bytestring=svg_string.encode('utf-8'), output_width=200, output_height=200, background_color="white")
        # OPTIMIZED:
        png_data = cairosvg.svg2png(bytestring=svg_string.encode('utf-8'), output_width=160, output_height=160, background_color="white")
        return np.array(Image.open(io.BytesIO(png_data)).convert('L'))
    except Exception: 
        return None

def calculate_ted(generated_svg, target_svg):
    def extract_tag_sequence(svg_string):
        try:
            root = ET.fromstring(svg_string)
            return [elem.tag.split('}')[-1] for elem in root.iter()]
        except Exception: 
            return []
    tags_gen = extract_tag_sequence(extract_svg(generated_svg))
    tags_tgt = extract_tag_sequence(target_svg)
    if not tags_gen or not tags_tgt: return 1000.0
    return float(editdistance.eval(tags_gen, tags_tgt))

def visual_similarity_reward(prompts, completions, ground_truths, **kwargs):
    rewards = []
    for completion, tgt_svg in zip(completions, ground_truths):
        # Safety catch: TRL sometimes passes strings, sometimes lists of dicts
        gen_text = completion[0]['content'] if isinstance(completion, list) else completion
        try:
            gen_img = render_to_numpy(gen_text)
            tgt_img = render_to_numpy(tgt_svg)
            if gen_img is None or tgt_img is None:
                rewards.append(0.0)
                continue
            ssim_score = ssim(tgt_img, gen_img, data_range=255)
            edges_tgt = cv2.Canny(tgt_img, 100, 200) > 0
            edges_gen = cv2.Canny(gen_img, 100, 200) > 0
            tp = np.logical_and(edges_tgt, edges_gen).sum()
            fp = np.logical_and(np.logical_not(edges_tgt), edges_gen).sum()
            fn = np.logical_and(edges_tgt, np.logical_not(edges_gen)).sum()
            f1 = (2 * tp / (2 * tp + fp + fn)) if (tp + fp + fn) > 0 else 0.0
            
            final_score = ((ssim_score + f1) / 2.0) * 0.85
            rewards.append(float(final_score))
        except Exception: 
            rewards.append(0.0)
    return rewards

def structural_reward(prompts, completions, ground_truths, **kwargs):
    rewards = []
    for completion, tgt_svg in zip(completions, ground_truths):
        gen_text = completion[0]['content'] if isinstance(completion, list) else completion
        try:
            ted = calculate_ted(gen_text, tgt_svg)
            s_score = np.exp(-ted / 25.0)
            rewards.append(float(s_score * 0.12))
        except Exception: 
            rewards.append(0.0)
    return rewards

def syntax_survival_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        gen_text = completion[0]['content'] if isinstance(completion, list) else completion
        try:
            # Extract the raw SVG from the markdown blocks
            clean_svg = extract_svg(gen_text).strip()
            if clean_svg.startswith("<svg") and clean_svg.endswith("</svg>"): 
                rewards.append(0.1)
            else: 
                rewards.append(-1.0)
        except Exception: 
            rewards.append(-1.0)
    return rewards

# ==========================================
# DATA FORMATTING
# ==========================================
def format_for_grpo(example):
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"
    # Exact match to SFT prompt!
    prompt = f"{open_comment} Description: {example['prompt']} {close_comment}\n```xml\n"
    return {
        "prompt": [{"role": "user", "content": prompt}], 
        "ground_truths": example["svg"]
    }

# ==========================================
# MAIN TRAINING LOOP
# ==========================================
if __name__ == "__main__":
    MODEL_ID = "./sft-merged-model-3b" 
    
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map={"": local_rank}, 
        torch_dtype=torch.float16,
    )
    model.warnings_issued = {}
    model.config.use_cache = True
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.padding_side = "left" 
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id
    
    # Satisfy the GRPOTrainer chat_template requirement 
    tokenizer.chat_template = "{% for message in messages %}{{ message['content'] }}{% endfor %}"

    # The New GRPO LoRA Adapter (Matches SFT size!)
    grpo_peft_config = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )

    # Load the highly optimized dataset
    train_optimized_filepath = "/kaggle/input/datasets/jef9921/train-optimized/train_optimized.parquet"
    raw_dataset = load_dataset("parquet", data_files=train_optimized_filepath, split="train")
    
    # We select 1000 samples. GRPO learns heavily from trial-and-error, so 1k is plenty!
    # grpo_dataset = raw_dataset.map(format_for_grpo, remove_columns=raw_dataset.column_names).select(range(1000))
    
    # We select 500 samples due to training time constraints. GRPO learns heavily from trial-and-error, so 500 is good enough! (Not in this case)
    grpo_dataset = raw_dataset.map(format_for_grpo, remove_columns=raw_dataset.column_names).select(range(500))

    rl_config = GRPOConfig(
        output_dir="./svg-phase2-rl",
        
        # Advanced Training Dynamics 
        learning_rate=1e-5,               # Keep this low! RL needs a small LR.
        lr_scheduler_type="cosine",       # Smooth decay
        warmup_steps=25,                  # 10% of max_steps for a gentle start
        weight_decay=0.01,                # Regularization
        optim="paged_adamw_8bit",         # CRITICAL: Brings back SFT VRAM savings

        # # Went Down from 32 -> 16, Not enough time
        # # Went Down From 16 - 8, ran into DDP-timeout error, GPU1 was probably waiting for GPU0 for longer than 30 minutes; Crashed
        # # 8 Generations divided by 2 GPUs = 4 generations per GPU (VRAM Safe!)
        # num_generations=8, 
        # per_device_train_batch_size=1, 
        # gradient_accumulation_steps=4,    
        
        # max_prompt_length=256, 
        # # max_completion_length=1024, 
        # max_completion_length=768, 
        # max_steps=250, 

        # SPEED TWEAKS 
        num_generations=4, 
        per_device_train_batch_size=1, 
        # gradient_accumulation_steps=4,  # Drops time-per-step by 50%
        gradient_accumulation_steps=2,  # Drops time-per-step by 50% again
        
        max_prompt_length=256, 
        max_completion_length=2048, 
        max_steps=125,                  # Caps total time to ~11 hours
        
        fp16=True,  
        bf16=False, 
        logging_steps=10,
        
        # CRITICAL DDP FIXES
        # The Multi-GPU Safety Net
        # Tells PyTorch to wait up to 1.5 hours (5400 seconds) for the other GPU
        # instead of killing the run after 30 minutes.
        ddp_timeout=5400,
        ddp_find_unused_parameters=False,
        use_vllm=False, 

        # THE CHECKPOINTING TWEAKS 
        save_strategy="steps",
        save_steps=25,
        save_total_limit=1,  # CRITICAL: Keeps only the most recent save to prevent Kaggle disk OOM!
    )

    grpo_trainer = GRPOTrainer(
        model=model,
        reward_funcs=[visual_similarity_reward, structural_reward, syntax_survival_reward], 
        args=rl_config,
        train_dataset=grpo_dataset,
        peft_config=grpo_peft_config, 
        # Pass our patched tokenizer to the trainer! 
        processing_class=tokenizer
    )

    # THE RESUME LOGIC 
    last_checkpoint = None
    if os.path.isdir(rl_config.output_dir):
        last_checkpoint = get_last_checkpoint(rl_config.output_dir)

    print(f"[GPU {local_rank}] Starting GRPO Training...")
    
    if last_checkpoint is not None:
        print(f"[GPU {local_rank}] Resuming from crashed checkpoint: {last_checkpoint}")
        grpo_trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        print(f"[GPU {local_rank}] Starting fresh training run.")
        grpo_trainer.train()
    
    if local_rank == 0:
        grpo_trainer.save_model("./grpo-lora-adapter-final")
        print("GRPO Phase 2 Complete!")

Overwriting train_grpo.py


In [ ]:
# !accelerate launch --num_processes 2 train_grpo.py

In [ ]:
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel

# print("Loading base model to CPU.")
# # Load the base model strictly to the CPU to avoid the 8GB VRAM limit
# base_model = AutoModelForCausalLM.from_pretrained(
#     "unsloth/Qwen2.5-Coder-3B", 
#     torch_dtype=torch.float16,
#     device_map="cpu", 
# )

# print("Attaching GRPO LoRA adapter.")
# # Attach your Phase 1 adapter
# model = PeftModel.from_pretrained(
#     base_model,
#     # "./grpo-lora-adapter-final",
#     "/kaggle/input/datasets/jef9921/grpo-lora-adapter-final/grpo-lora-adapter-final",
#     device_map="cpu",
# )

# print("Merging weights (this will take a while).")
# # Fuse the LoRA weights directly into the base model's linear layers
# merged_model = model.merge_and_unload()

# print("Saving merged model to disk.")
# # Export the standalone Hugging Face model
# merged_model.save_pretrained("./sft-merged-model-3b-grpo")

# # tokenizer = AutoTokenizer.from_pretrained("./sft-lora-adapter-3B")
# tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/datasets/jef9921/grpo-lora-adapter-final/grpo-lora-adapter-final")
# tokenizer.save_pretrained("./sft-merged-model-3b-grpo")

# print("Merge complete! Safe for Outlines inference.")

In [4]:
%%writefile inference.py
import os
import argparse
import pandas as pd
import torch
import io
import cairosvg
import numpy as np
from PIL import Image
import xml.etree.ElementTree as ET
import warnings
import re

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModel
import outlines
from outlines.types import CFG

import time
from datetime import datetime
import sys

warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# ARGUMENT PARSER 
parser = argparse.ArgumentParser()
parser.add_argument("--gpu", type=int, default=0, help="GPU ID to use (0 or 1)")
parser.add_argument("--part", type=int, default=0, help="Part ID to use")
parser.add_argument("--start", type=int, default=0, help="Start index of the dataset")
parser.add_argument("--end", type=int, default=500, help="End index of the dataset")
args = parser.parse_args()

print(f"[GPU {args.gpu}] Initializing local inference.")

# Force the script to exclusively use the assigned GPU
device = f"cuda:{args.gpu}" 

# ==========================================
# LOAD MAIN MODEL (16-BIT PRECISION)
# ==========================================
print(f"[GPU {args.gpu}] Loading 3B model in 16-bit (Float16).")
hf_model = AutoModelForCausalLM.from_pretrained(
    "./sft-merged-model-3b-grpo", 
    torch_dtype=torch.float16, 
    device_map=device,
)

hf_model.generation_config.max_length = None
hf_tokenizer = AutoTokenizer.from_pretrained("./sft-merged-model-3b-grpo")
generator = outlines.from_transformers(hf_model, hf_tokenizer)

official_kaggle_grammar = CFG("""
    ?start: WS? svg WS?
    svg: "<svg" WS? ATTR_LIST ">" WS? elements "</svg>"
    elements: (element | TEXT)*
    element: "<" TAG WS? ATTR_LIST "/>" WS? | "<" TAG WS? ATTR_LIST ">" WS? elements "</" TAG ">" WS?
    TAG: "svg" | "g" | "path" | "rect" | "circle" | "ellipse" | "line" | "polyline" | "polygon" | "defs" | "use" | "symbol" | "clipPath" | "mask" | "linearGradient" | "radialGradient" | "stop" | "text" | "tspan" | "title" | "desc" | "style" | "pattern" | "marker" | "filter"
    ATTR_LIST: (/[a-zA-Z0-9_:-]+/ WS? "=" WS? /"[^"]*"/ WS?)*
    TEXT: /[^<]+/
    WS: /[ \\t\\n\\r]+/
""")

# ==========================================
# LOAD SIGLIP JUDGE (16-BIT PRECISION)
# ==========================================
print(f"[GPU {args.gpu}] Loading SigLIP Judge.")
siglip_id = "google/siglip-so400m-patch14-384"
processor = AutoProcessor.from_pretrained(siglip_id)

judge = AutoModel.from_pretrained(
    siglip_id, 
    torch_dtype=torch.float16 
).to(device)
judge.eval() 

# ==========================================
# HELPER FUNCTIONS
# ==========================================
def extract_svg(text):
    match = re.search(r'(<svg.*?</svg>)', text, re.IGNORECASE | re.DOTALL)
    return match.group(1) if match else text

def heal_svg(raw_svg):
    if raw_svg.strip().endswith("</svg>"): return raw_svg
    last_closed_idx = raw_svg.rfind("/>")
    if last_closed_idx != -1:
        return raw_svg[:last_closed_idx + 2] + "\n</svg>"
    return raw_svg

def is_kaggle_compliant(svg_string):
    if len(svg_string) > 16000: return False
    try:
        root = ET.fromstring(svg_string)
        if root.tag.split('}')[-1] != 'svg': return False
    except ET.ParseError: return False
    if svg_string.count("<path") > 256: return False
    return True

def render_to_numpy(svg_string):
    try:
        png_data = cairosvg.svg2png(bytestring=svg_string.encode('utf-8'), output_width=256, output_height=256, background_color="white")
        return np.array(Image.open(io.BytesIO(png_data)).convert('L'))
    except: return None

def select_best_svg(prompt_text, candidate_svgs):
    valid_images, valid_svgs = [], []
    for i, raw_svg in enumerate(candidate_svgs):
        clean_svg = heal_svg(extract_svg(raw_svg))
        if not is_kaggle_compliant(clean_svg): continue
        img = render_to_numpy(clean_svg)
        if img is None: continue
        valid_images.append(Image.fromarray(img).convert('RGB'))
        valid_svgs.append(clean_svg) 
            
    if not valid_images: return "<svg></svg>"
    if len(valid_images) == 1: return valid_svgs[0]

    inputs = processor(
        text=[prompt_text], images=valid_images, return_tensors="pt", 
        padding="max_length", truncation=True, max_length=64    
    ).to(device)
    
    with torch.no_grad():
        scores = judge(**inputs).logits_per_image.squeeze().cpu().numpy()
        
    if scores.ndim == 0: return valid_svgs[0]
    return valid_svgs[scores.argmax()]

# ==========================================
# INFERENCE LOOP
# ==========================================
if __name__ == "__main__":
    # SLICE THE DATASET 
    # full_df = pd.read_csv("dl-spring-2026-svg-generation/test.csv") 
    full_df = pd.read_csv("/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv") 
    test_df = full_df.iloc[args.start:args.end].copy()
    
    csv_filename = f"./submission-3b-2048-grpo_part_{args.part}.csv"
    NUM_CANDIDATES = 2
    MAX_CONTEXT_WINDOW = 2048
    
    print(f"\n[GPU {args.gpu}] Starting Inference on {len(test_df)} prompts (Rows {args.start} to {args.end})...")
    sys.stdout.flush() # Force write to file!

    for idx in range(len(test_df)):
        # Start the stopwatch!
        start_time = time.time()
        current_time = datetime.now().strftime("%H:%M:%S")
        
        row = test_df.iloc[idx]
        print(f"\n[{current_time}] [GPU {args.gpu}] [{idx+1}/{len(test_df)}] Generating ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        sys.stdout.flush() 
        
        open_comment = "<!-" + "-"
        close_comment = "-" + "->"
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        
        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10

        candidates = [] 
        
        try:
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,
                num_return_sequences=NUM_CANDIDATES, 
            )
            
            if isinstance(raw_candidate, list):
                candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] [GPU {args.gpu}] Generation failed: {e}")
            sys.stdout.flush()
            
        torch.cuda.empty_cache() 
            
        best_svg = select_best_svg(row["prompt"], candidates)

        print(f"Best SVG: {best_svg}")
        sys.stdout.flush()
        
        result_df = pd.DataFrame([{"id": row["id"], "svg": best_svg}])
        if idx == 0 or not os.path.exists(csv_filename):
            result_df.to_csv(csv_filename, index=False, mode='w')
        else:
            result_df.to_csv(csv_filename, index=False, mode='a', header=False)
            
        torch.cuda.empty_cache()

        # Stop the stopwatch and log the duration!
        elapsed_time = time.time() - start_time
        print(f"[{datetime.now().strftime('%H:%M:%S')}] [GPU {args.gpu}] Finished prompt {idx+1} in {elapsed_time:.2f} seconds.")
        sys.stdout.flush()

    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] [GPU {args.gpu}] Local inference complete! Saved to {csv_filename}")
    sys.stdout.flush()

Overwriting inference.py


In [6]:
import pandas as pd
from transformers import AutoTokenizer

# Load your local SFT-trained tokenizer
# You can point this to "./sft-merged-model-3b" or "Qwen/Qwen2.5-Coder-3B"
TOKENIZER_PATH = "./sft-merged-model-3b" 

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

# Load the highly optimized dataset
PARQUET_PATH = "/kaggle/input/datasets/jef9921/train-optimized/train_optimized.parquet"
print(f"Loading {PARQUET_PATH}...")
df = pd.read_parquet(PARQUET_PATH)

print("Calculating exact token lengths for 49,043 SVGs (this takes about 60 seconds)...")

# Encode the SVGs and count the tokens
# We add_special_tokens=False because we only care about the SVG payload length
lengths = df['svg'].apply(lambda x: len(tokenizer.encode(str(x), add_special_tokens=False)))

# Extract the critical metrics
max_len = lengths.max()
p99_len = lengths.quantile(0.99)
p95_len = lengths.quantile(0.95)
avg_len = lengths.mean()

print("\n======================================")
print("       SVG TOKEN LENGTH STATS         ")
print("======================================")
print(f"Absolute Max Length: {max_len} tokens")
print(f"99th Percentile:     {p99_len:.0f} tokens")
print(f"95th Percentile:     {p95_len:.0f} tokens")
print(f"Average Length:      {avg_len:.0f} tokens")
print("======================================")

Loading tokenizer...
Loading /kaggle/input/datasets/jef9921/train-optimized/train_optimized.parquet...
Calculating exact token lengths for 49,043 SVGs (this takes about 60 seconds)...


Token indices sequence length is longer than the specified maximum sequence length for this model (33864 > 32768). Running this sequence through the model will result in indexing errors



       SVG TOKEN LENGTH STATS         
Absolute Max Length: 44909 tokens
99th Percentile:     4988 tokens
95th Percentile:     2503 tokens
Average Length:      1173 tokens


In [ ]:
# %%bash
# echo "Starting Dual-GPU Inference..."
# echo "Live output is being routed to part0_log.txt and part1_log.txt"

# # The '> filename.txt 2>&1' part forces all prints and errors into a text file
# python -u inference.py --gpu 0 --part 0 --start 0 --end 250 > part0_log.txt 2>&1 &
# python -u inference.py --gpu 1 --part 1 --start 250 --end 500 > part1_log.txt 2>&1 &

# wait

# echo "Both GPUs have finished generating!"

In [ ]:
# %%bash
# echo "Starting Dual-GPU Inference..."
# echo "Live output is being routed to part2_log.txt and part3_log.txt"

# # The '> filename.txt 2>&1' part forces all prints and errors into a text file
# python -u inference.py --gpu 0 --part 2 --start 500 --end 750 > part2_log.txt 2>&1 &
# python -u inference.py --gpu 1 --part 3 --start 750 --end 1000 > part3_log.txt 2>&1 &

# wait

# echo "Both GPUs have finished generating!"

In [1]:
import pandas as pd

print("Merging GPU results...")

# Load both halves generated by the bash script
df_part0 = pd.read_csv("/kaggle/input/datasets/jef9921/submission-split/submission-3b-2048-grpo_part_0.csv")
df_part1 = pd.read_csv("/kaggle/input/datasets/jef9921/submission-split/submission-3b-2048-grpo_part_1.csv")
df_part2 = pd.read_csv("/kaggle/input/datasets/jef9921/submission-split/submission-3b-2048-grpo_part_2.csv")
df_part3 = pd.read_csv("/kaggle/input/datasets/jef9921/submission-split/submission-3b-2048-grpo_part_3.csv")

# Combine them sequentially
final_submission = pd.concat([df_part0, df_part1, df_part2, df_part3], ignore_index=True)

# Save to the final 'submission.csv' required by the competition
final_submission.to_csv("./submission-3b-2048-grpo.csv", index=False)

print(f"Merge successful! Total rows evaluated: {len(final_submission)}")
final_submission.head()

Merging GPU results...
Merge successful! Total rows evaluated: 1000


,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg version=""1.1"" xmlns=""http://www.w3.org/20..."
1,6eede943219547c22ac56085027d33cc,"<svg width=""300"" height=""50"" xmlns=""http://www..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg width=""50"" height=""50"" viewBox=""0 0 80 80..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg width=""300"" height=""200"">\n <rect x=""50""..."
4,600464e4d92c75338462271a09b3f176,"<svg width=""200"" height=""200"">\n <polygon poi..."


In [3]:
# Identify the missing rows from Pass 1
sub_df = pd.read_csv("./submission-3b-2048-grpo.csv")
test_df = pd.read_csv("/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs.")

Found 4 blank SVGs.


In [5]:
%%writefile train_sft_epoch_2.py
import os
import torch
from unsloth import FastLanguageModel

import transformers.utils.hub
transformers.utils.hub.TRANSFORMERS_CACHE = os.getenv("HF_HOME", "~/.cache/huggingface/hub")
os.environ["WANDB_DISABLED"] = "true"

from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

if __name__ == "__main__":
    train_optimized_filepath = "/kaggle/input/datasets/jef9921/train-optimized/train_optimized.parquet"
    raw_dataset = load_dataset("parquet", data_files=train_optimized_filepath, split="train")
    
    # THE DATA ALPHA STRATEGY 
    # Instead of repeating the first 25,000, we feed it the REMAINING 24,043 SVGs!
    # The model learns entirely new concepts it has never seen before.
    dataset_split = raw_dataset.select(range(25000, 49043)).train_test_split(test_size=500, seed=42)
    train_dataset = dataset_split["train"]
    eval_dataset = dataset_split["test"]
    
    # Load the 16-bit merged model 
    MODEL_ID = "./sft-merged-model-3b" 
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=2048,
        dtype=torch.float16, 
        load_in_4bit=True,
    )
    
    # We apply a fresh LoRA adapter to capture these new, refined details
    model = FastLanguageModel.get_peft_model(
        model, 
        r=32, 
        lora_alpha=64, 
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
        lora_dropout=0, 
        bias="none",
        use_gradient_checkpointing=True,
        random_state=42, # Changed seed to shake up the data ordering
    )
    
    EOS = tokenizer.eos_token
    
    # Format the dataset for SFT by combining prompt and optimized SVG into a single text field
    def format_for_sft(example):
        open_comment = "<!-" + "-"
        close_comment = "-" + "->"
        text = f"{open_comment} Description: {example['prompt']} {close_comment}\n```xml\n{example['svg']}\n```{EOS}"
        return {"text": text}
    
    train_dataset = train_dataset.map(format_for_sft)
    eval_dataset = eval_dataset.map(format_for_sft)
    
    sft_config = SFTConfig(
        output_dir="./svg-phase2-sft-epoch2", # Save to a new folder!
        dataset_text_field="text",
        max_seq_length=2048,
        
        # Packing Enabled to Squeeze More Learning Signal
        packing=True, 
        
        num_train_epochs=1, 
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8, 
        
        # THE WARM RESTART DYNAMICS
        learning_rate=5e-5,          # Dropped from 2e-4 to gently refine the weights
        lr_scheduler_type="cosine", 
        warmup_ratio=0.05,           # 5% warmup for the new curve
        weight_decay=0.01,          
        neftune_noise_alpha=5,      
        
        fp16=True,  
        bf16=False, 
        optim="paged_adamw_8bit", 
        logging_steps=50,
        
        eval_strategy="steps",
        eval_steps=200, 
        
        save_strategy="steps", 
        save_steps=200,
        load_best_model_at_end=True,         
        metric_for_best_model="eval_loss",   
        greater_is_better=False,             
        save_total_limit=2,
        ddp_find_unused_parameters=False,
    )
    
    sft_trainer = SFTTrainer(
        model=model, 
        tokenizer=tokenizer, 
        train_dataset=train_dataset, 
        eval_dataset=eval_dataset,
        args=sft_config
    )
    
    print("Starting SFT Epoch 2 (Part 2) with Merged Qwen2.5-Coder-3B...")
    sft_trainer.train()
    
    # Save the new adapter!
    model.save_pretrained("./sft-lora-adapter-3B-epoch2")
    tokenizer.save_pretrained("./sft-lora-adapter-3B-epoch2")

Writing train_sft_epoch_2.py


In [ ]:
# !accelerate launch --num_processes 2 train_sft_epoch_2.py